# CSCI 447/547 Hackathon 6: Support Vector Machines and Decision Trees

This notebook is designed to be started during class and continued as a take-home activity

## How to use hackathon notebooks:

If the topic covered in a hackathon is new to you, work through the cells in order and read the explanation before running each code cell. You do not need to understand every detail of the hackathon on the first pass, instead, focus on the approach we take:

1. **Look at the data**
2. **Separate inputs from the target we want to predict**
3. **Split the data so we can test whether the model generalizes**
4. **Fit a model using the training data**
5. **Make predictions and evaluate them**
6. **Improve the model carefully <u>without</u> using the final evaluation data to make decisions**

We will work through the workbook together as a class. Pause before important code cells and think about what you expect to see. Please ask Lucy or a TA questions any time a term or line of code is unfamiliar.

After class, continue from wherever you left off. The existing explanations and code will be there to guide you through the process. Complete the marked answer sections, run every cell, and explain what the results mean in language that makes sense for you. Hackathons will not be graded, they are only to help you, and you will get out of them what you put into them.

<h4><span style="color:red">The goal of this notebook is NOT to memorize every function. It is to identify the processes we use in machine learning and be able to reuse them.</span></h4>

##### In this exercise, you will train support vector machines and decision trees, explore how their hyperparameters shape their decision boundaries, and compare them to logistic regression.

---

### Google Colab Instructions

If you are using Google Colab <a href="https://colab.research.google.com/github/lucywowen/csci547_ML/blob/main/examples/hackathon6-SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Google Colab"/></a> you do not need to download anything: every dataset in this notebook is built into `scikit-learn`.

---

### Local (VSCode/Jupyter Lab) Version

To get the latest version of this notebook **without overwriting your work on other hackathons**, open a terminal in your cloned `csci547_ML` folder and run:

```bash
git fetch
git checkout origin/main -- examples/hackathon6-SVM.ipynb
```

`git fetch` downloads the latest version of the repository without changing any of your files, then `git checkout` replaces only this notebook. *(This overwrites any changes you have made to this notebook, so if you have already started it, make a copy first!)*

Make sure that your `uv` virtual environment is enabled.

---

## 1. Support Vector Machines

A **Support Vector Machine (SVM)** takes a geometric approach to classification: it finds the boundary that separates the classes with the widest possible **margin**, the empty "street" between them.

- Real data is rarely perfectly separable, so an SVM allows some points inside the margin (or even on the wrong side), but charges a penalty for each one. The hyperparameter **C** controls this trade-off: a large $C$ makes these violations expensive, while a small $C$ tolerates them in exchange for a wider margin.
- The training points that sit on or inside the margin are called **support vectors**. They are the *only* points that determine the boundary: you could move any other point (without crossing the margin) and the SVM wouldn't change at all.

### 1.1 Load the Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_theme(style='white')

from sklearn import datasets
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

### 1.2 A Helper for Plotting Decision Boundaries

We'll draw a lot of decision boundaries, so here's a helper function (handled for you). It colors each region of the plot by the class the model predicts there, then draws the data points on top. With `show_margin=True`, it also draws an SVM's margin as dashed lines and circles its support vectors in red.

In [ ]:
def plot_decision_boundary(model, X, y, ax, title='', show_margin=False):
    """Plots a fitted model's decision regions for 2D data X with labels y on the axis ax."""
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    grid = np.c_[xx.ravel(), yy.ravel()]

    # color each point of the grid by the class the model predicts there
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, cmap=plt.cm.viridis, alpha=0.3)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.viridis, edgecolors='k', s=30)

    if show_margin:
        # the boundary (solid line) and the edges of the margin (dashed lines)
        f = model.decision_function(grid).reshape(xx.shape)
        ax.contour(xx, yy, f, levels=[-1, 0, 1], colors='k', linestyles=['--', '-', '--'])
        sv = model.support_vectors_
        ax.scatter(sv[:, 0], sv[:, 1], s=120, facecolors='none', edgecolors='r', linewidths=1.5)

    ax.set(xticks=[], yticks=[], title=title)

### 1.3 Margins and C

Let's start with two blobs of points that are *almost* separable by a straight line, and fit a linear SVM to them (handled for you). scikit-learn's `SVC` (support vector classifier) works just like the other scikit-learn models we've used. Its `kernel` argument chooses the kernel (more on kernels in Section 2), and its `C` argument sets $C$. Once it's fit, its `support_vectors_` attribute holds the support vectors, one row per support vector.

In [ ]:
blobs_X, blobs_y = datasets.make_blobs(n_samples=100, centers=2, cluster_std=1.4, random_state=3)

model = SVC(kernel='linear', C=1)
model.fit(blobs_X, blobs_y)
print(f'C = 1: {len(model.support_vectors_)} support vectors')

fig, ax = plt.subplots(figsize=(7, 5))
plot_decision_boundary(model, blobs_X, blobs_y, ax, title='Linear SVM, C = 1', show_margin=True)
plt.show()

### Think-Pair-Share 1: The C Hyperparameter
**Context:** $C$ sets how much the SVM is penalized for each point inside the margin or on the wrong side.

1. **Think:** If $C$ is very small (say $C = 0.01$), what will the margin look like, and will many or few points end up as support vectors? What about a very large $C$ (say $C = 100$)? Which one is more likely to overfit?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** In regularized logistic regression, a larger $\lambda$ meant a *stronger* penalty on the weights (a simpler model). Discuss with your group: how are $C$ and $\lambda$ related?

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

Let's check your hypotheses with three values of $C$ (handled for you):

In [ ]:
C_values = [0.01, 1, 100]
fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(16, 5))

for ax, C in zip(axs, C_values):
    model = SVC(kernel='linear', C=C)
    model.fit(blobs_X, blobs_y)
    plot_decision_boundary(model, blobs_X, blobs_y, ax, title=f'C = {C}', show_margin=True)
    print(f'C = {C}: {len(model.support_vectors_)} support vectors')

plt.show()

As $C$ grows, the margin narrows and fewer points end up inside it. With $C = 100$, the boundary depends on just 4 points: if one of them were an outlier or mislabeled, the whole boundary would follow it.

## 2. Kernels: When a Straight Line Won't Do

The real power of SVMs is their ability to classify data that is **not** linearly separable, like a circle inside a circle. Let's create some, and split it into training and test sets.

In [ ]:
circles_X, circles_y = datasets.make_circles(n_samples=500, noise=0.25, factor=0.5, random_state=123)

circles_X_train, circles_X_test, circles_y_train, circles_y_test = train_test_split(
    circles_X, circles_y, test_size=0.3, random_state=42, stratify=circles_y)

plt.figure(figsize=(7, 5))
plt.scatter(circles_X_train[:, 0], circles_X_train[:, 1], c=circles_y_train, cmap=plt.cm.viridis, edgecolors='k')
plt.title('Circles (Training Set)')
plt.show()

No straight line can separate these classes, so SVMs use a **kernel**. In class, we discussed **landmarks** and the Gaussian, or **RBF**, kernel, which measures how *similar* a point $\mathbf{x}$ is to a landmark $\mathbf{l}$:

$$\text{similarity}(\mathbf{x}, \mathbf{l}) = \exp\Big(-\frac{\Vert\mathbf{x}-\mathbf{l}\Vert^2}{2\sigma^2}\Big) = \exp\big(-\gamma \Vert\mathbf{x}-\mathbf{l}\Vert^2\big)$$

The similarity is 1 when $\mathbf{x}$ is right on the landmark, and falls towards 0 as $\mathbf{x}$ moves away. The RBF kernel uses every training point as a landmark, which lets the SVM draw curved boundaries. $\sigma^2$ sets how wide each similarity "bump" is. scikit-learn uses $\gamma$ (`gamma`) instead, where $\gamma = \frac{1}{2\sigma^2}$: a **small** $\sigma^2$ is a **large** $\gamma$.

Let's compare logistic regression with linear and RBF SVMs, using 5-fold cross-validation on the training set (handled for you):

In [ ]:
models = {
    'Logistic regression': LogisticRegression(),
    'SVM, linear kernel': SVC(kernel='linear'),
    'SVM, RBF kernel': SVC(kernel='rbf'),
}

print('Cross-validated mean accuracy:')
for name, model in models.items():
    scores = cross_val_score(model, circles_X_train, circles_y_train, cv=5, scoring='accuracy')
    print(f'\t{name:22s}: {scores.mean():.3f}')

Both linear models are stuck near 50%, no better than guessing. Only the RBF kernel can bend its boundary around the inner circle.

### Think-Pair-Share 2: The γ Hyperparameter
**Context:** A **large** $\gamma$ (small $\sigma^2$) gives a **narrow** similarity bump, and a **small** $\gamma$ gives a **wide** one.

1. **Think:** With a very large $\gamma$, how many training points will a new point be "similar" to? Sketch what you expect the SVM's boundary on the circles data to look like for $\gamma = 0.01$, $\gamma = 1$, and $\gamma = 100$. Which will underfit, and which will overfit?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Compare sketches with your group. How is an RBF SVM with a huge $\gamma$ similar to KNN with $k = 1$ from Hackathon 1?

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

### 2.1 Coding Exercise: How γ Changes the Boundary

Time to check your sketches! From here on, you will be writing more of the code yourself. Every function you need has been used either earlier in this notebook or in a previous hackathon, so when you're not sure how to write something, look back at those cells.

**What you need to write:** Inside the loop, do what the loop over $C$ in Section 1.3 did, but with an RBF SVM: train it on the circles **training** set using the loop's current value of `gamma` (`SVC` takes a `gamma` argument), and draw its decision boundary on that loop's subplot, `ax`. Each subplot's title should show its value of $\gamma$ along with the SVM's accuracy on both the training set and the test set, so you can see underfitting and overfitting side by side. (Every scikit-learn classifier has a `.score()` method that computes its accuracy.)

*Expected (approx) train/test accuracy:* 0.63/0.60, 0.85/0.85, and 0.94/0.81.

In [ ]:
gamma_values = [0.01, 1, 100]
fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(16, 5))

for ax, gamma in zip(axs, gamma_values):
    # --- YOUR CODE HERE ---

plt.show()

So, to sum it up: a **large $\gamma$** (small $\sigma^2$) **overfits**, because each training point only influences its immediate neighborhood. A **small $\gamma$** (large $\sigma^2$) **underfits**, because every training point influences everything. $C$ and $\gamma$ interact, so in practice we tune them together.

## 3. Real Data: Handwritten Digits

scikit-learn includes a small version of the famous MNIST dataset of handwritten digits: 1,797 images, each 8×8 pixels. Each of the 64 pixels is a feature, with a value from 0 (white) to 16 (black). We'll set aside 25% of the images as a generalization set.

In [ ]:
digits = datasets.load_digits()
X_train, X_gen, y_train, y_gen = train_test_split(digits.data, digits.target, test_size=0.25, random_state=42, stratify=digits.target)

fig, axs = plt.subplots(nrows=1, ncols=10, figsize=(16, 2.5))
for ax, image, label in zip(axs, digits.images, digits.target):
    ax.imshow(image, cmap='gray_r')
    ax.set(xticks=[], yticks=[], title=f'Label: {label}')
plt.show()

### Think-Pair-Share 3: Multi-Class SVMs
**Context:** An SVM finds a single boundary that separates **two** classes. The digits dataset has **ten** classes.

1. **Think:** How could you combine several *binary* (two-class) SVMs to classify the ten digits? Describe at least one strategy, and how many SVMs it would need.

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Compare strategies with your group. scikit-learn's `SVC` uses one-vs-one. How many SVMs does it train for ten classes? What is one advantage and one disadvantage of this strategy?

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

### 3.1 Coding Exercise: Tune the SVM with GridSearchCV

Let's tune $C$, $\gamma$, and the kernel together with `GridSearchCV` (from Hackathon 0), which tries every combination of hyperparameters using cross-validation on the training set. A linear kernel has no $\gamma$, so we give `GridSearchCV` a **list** of grids, one per kernel. (Otherwise, it would waste time retraining the linear SVM with every value of $\gamma$.)

**What you need to write:** Add a second grid to `param_grid` for the `'rbf'` kernel, covering every combination of `C` = 0.1, 1, 10, and 100 with `gamma` = 0.0001, 0.001, 0.01, and 0.1. Then build a `GridSearchCV` named `grid` that searches an `SVC()` over `param_grid`, judging each combination by its 5-fold cross-validated accuracy, and fit it to the training set. This takes a few seconds.

*Expected (approx):* best parameters `{'C': 10, 'gamma': 0.001, 'kernel': 'rbf'}`, with a cross-validated accuracy of 0.9918.

In [ ]:
param_grid = [
    {'kernel': ['linear'], 'C': [0.01, 0.1, 1, 10]},
    # --- YOUR CODE HERE ---
]

# --- YOUR CODE HERE ---

print('Best parameters:', grid.best_params_)
print(f'Best cross-validated accuracy: {grid.best_score_:.4f}')

### 3.2 Evaluate Generalization

After the search, `GridSearchCV` automatically retrains the best model on the whole training set, so `grid` can make predictions like any other model. Let's evaluate it on the generalization set, and look at the handful of digits it got wrong (handled for you). Would you have gotten them right?

In [ ]:
gen_pred = grid.predict(X_gen)
print(f'Generalization accuracy: {accuracy_score(y_gen, gen_pred):.4f}')

ConfusionMatrixDisplay.from_predictions(y_gen, gen_pred)
plt.title('Tuned SVM Confusion Matrix (Generalization Set)')
plt.show()

wrong = np.where(gen_pred != y_gen)[0]
fig, axs = plt.subplots(nrows=1, ncols=len(wrong), figsize=(2.5 * len(wrong), 3))
for ax, i in zip(np.atleast_1d(axs), wrong):
    ax.imshow(X_gen[i].reshape(8, 8), cmap='gray_r')
    ax.set(xticks=[], yticks=[], title=f'True: {y_gen[i]}\nPredicted: {gen_pred[i]}')
plt.show()

## 4. Decision Trees

A **decision tree** learns a series of yes/no questions about the features (e.g. "is the petal length less than 2.45 cm?"). It's drawn upside down: the **root** at the top asks the first question, each **node** splits the samples into two groups, and each **leaf** at the bottom predicts the most common class of the training samples that end up there. Decision trees are popular because they are fast, need very little data preparation, and are **interpretable**: you can follow a tree's decisions by hand.

### Think-Pair-Share 4: Measuring Purity
**Context:** At each node, a decision tree chooses the question that makes the two resulting groups as *pure* (as close to a single class) as possible.

1. **Think:** A node contains 40 setosa, 40 versicolor, and 40 virginica irises. Split A sends all the setosa to the left and everything else to the right. Split B sends half of each species to the left and half to the right. Which split is better, and why? How could you measure the "purity" of a node with a single number?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Compare your purity measures with your group. What does yours give for a perfectly pure node? For a node with two classes split 50/50?

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

scikit-learn measures purity with the **Gini impurity**, which is 0 for a perfectly pure node and grows as the classes get more mixed. (`criterion='entropy'` uses entropy and information gain instead, as we discussed in class.)

### 4.1 A Tree for the Iris Data

Let's train a tree on the Iris dataset from Hackathon 0 and draw it with `plot_tree` (handled for you). Each box shows the question asked at that node, its Gini impurity, how many training samples reached it, how many of each class (`value`), and the class it would predict.

In [ ]:
iris = datasets.load_iris()
iris_X_train, iris_X_test, iris_y_train, iris_y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target)

iris_tree = DecisionTreeClassifier(random_state=42)
iris_tree.fit(iris_X_train, iris_y_train)

plt.figure(figsize=(16, 10))
plot_tree(iris_tree, feature_names=iris.feature_names, class_names=iris.target_names, filled=True)
plt.show()

print(classification_report(iris_y_test, iris_tree.predict(iris_X_test), target_names=iris.target_names))

**Pause point:** Which question did the tree ask first, at the root? Why do you think it chose that one?

Notice that the tree keeps splitting until every leaf is pure (`gini = 0.0`). When left **unpruned** like this, a decision tree is likely to overfit. We often **prune** it by tuning hyperparameters such as `max_depth`, the maximum number of questions in a row. This helpful [guide](https://towardsdatascience.com/how-to-tune-a-decision-tree-f03721801680) covers the other ones.

### 4.2 Coding Exercise: Tree Depth and Overfitting

How does a decision tree compare to the SVM on the digits data, and how deep should it be? Following our workflow, we'll measure this with cross-validation on the **training** set, not the generalization set.

**What you need to write:** By the end of the loop, `cv_scores` should hold the 5-fold cross-validated accuracy (on the digits training set) of a decision tree limited to each `max_depth` in `depths`, and `train_scores` should hold each tree's accuracy on the training data it was fit to. A tree limited to a depth of 3 is `DecisionTreeClassifier(max_depth=3, random_state=42)`, `cross_val_score()` handles the cross-validation (as in Section 2), and `.score()` gives the training accuracy. Then plot both lists against `depths` on the same axes, labeled so the legend can tell them apart (`plt.plot()` takes a `label` argument).

*Expected:* the training accuracy should climb to 1.00, while the cross-validated accuracy levels off around 0.85. Compare that to the tuned SVM!

In [ ]:
depths = range(1, 16)
train_scores = []
cv_scores = []

for depth in depths:
    # --- YOUR CODE HERE ---

plt.figure(figsize=(9, 5))
# --- YOUR CODE HERE ---
plt.xlabel('max_depth')
plt.ylabel('accuracy')
plt.title('Decision Tree Depth vs. Accuracy (Digits)')
plt.legend()
plt.show()

---

## Hackathon Takeaways & Synthesis

As your group finishes this notebook, collaborate to answer the following synthesis questions.

1. **Trade-off Analysis:**
On the digits data, the tuned RBF SVM reached about 99% accuracy, while the best decision tree only reached about 85%. However, a decision tree can be printed out and followed by hand, while an RBF SVM can't. Describe a real situation where you would choose the less accurate decision tree anyway. What would you be giving up?

    **ANSWER**

2. **Diagnosing Failures:**
A classmate's RBF SVM scores 100% on the training data but only 70% on new data. What is the problem? Which two hyperparameters should they change, and in which direction? What would the same problem look like for a decision tree, and how would they fix it there?

    **ANSWER**

3. **3-2-1 Summary:**
* **3** key mathematical or programming concepts your group solidified today:

    1. 

    2. 

    3. 

* **2** things you are still slightly confused about (we will address these in the next lecture!):

    1. 

    2. 

* **1** real-world scenario where you would choose an SVM, and one where you would choose a decision tree:

    1. 

### Things you *should* be able to answer:

1. How does the regularization hyperparameter ($C$) play a role in underfitting and overfitting in SVMs? How is $C$ related to the logistic regression hyperparameter $\lambda$?

2. How do the kernel hyperparameters ($C$, $\sigma^2$, $\gamma$) play a role in underfitting and overfitting in SVMs?

3. How does your choice in kernel (rbf vs linear) change based on the data?